In [1]:
!pip install datasets tiktoken tqdm pandas torch

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 555.1/555.1 kB 39.8 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 693.4/693.4 kB 39.8 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.5/4.5 MB 155.8 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 391.4 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.9/48.9 MB 251.9 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 801.2/801.2 kB 293.6 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11/11 [datasets]/11 [datasets]ce-hub]


In [2]:
import os
import math
import time
import json
import random
import inspect
from dataclasses import dataclass

import torch
import torch.nn as nn
import torch.nn.functional as F
from datasets import load_dataset
import tiktoken  # Overhauled tokenizer backend

# ─────────────────────────────────────────────────────────────
# 0.  GLOBAL HYPERPARAMETERS  (from MoEP codebase)
# ─────────────────────────────────────────────────────────────
VOCAB_SIZE        = 50257          # GPT-2 vocabulary
MASK_TOKEN_ID     = 50256          # reuse EOS as MASK (GPT-2 has no [MASK])
BLOCK_SIZE        = 512
BATCH_SIZE        = 16
GRAD_ACCUM_STEPS  = 1
EPOCHS            = 10
LEARNING_RATE     = 3e-4
LR_MIN            = LEARNING_RATE * 0.1
WARMUP_STEPS      = 800
WEIGHT_DECAY      = 0.1
GRAD_CLIP         = 1.0

# 1:15 causal-to-masked ratio → probability of picking a causal batch
CAUSAL_RATIO      = 1 / 16        # 1 causal : 15 masked

# Masking schedule: start at 30%, end at 15% (linear decay over training)
MASK_PROB_START   = 0.30
MASK_PROB_END     = 0.15

# Evaluation data paths (same as MoEP)
BLIMP_FAST_PATH   = "fast_eval/blimp_fast"
SUPP_FAST_PATH    = "fast_eval/supplement_fast"


# ─────────────────────────────────────────────────────────────
# 1.  SLIDING WINDOW ATTENTION  (from MSIT codebase)
# ─────────────────────────────────────────────────────────────

class SlidingWindowAttention(nn.Module):
    def __init__(self, dim: int, num_heads: int, window_size=None):
        super().__init__()
        assert dim % num_heads == 0, "dim must be divisible by num_heads"
        self.num_heads  = num_heads
        self.window_size = window_size
        self.head_dim   = dim // num_heads

        self.q_proj = nn.Linear(dim, dim, bias=False)
        self.k_proj = nn.Linear(dim, dim, bias=False)
        self.v_proj = nn.Linear(dim, dim, bias=False)
        self.o_proj = nn.Linear(dim, dim, bias=False)

    def forward(self, x: torch.Tensor,
                past_kv=None,
                use_cache: bool = False,
                bidirectional: bool = False):
        B, L, D = x.size()

        q = self.q_proj(x).view(B, L, self.num_heads, self.head_dim).transpose(1, 2)
        k = self.k_proj(x).view(B, L, self.num_heads, self.head_dim).transpose(1, 2)
        v = self.v_proj(x).view(B, L, self.num_heads, self.head_dim).transpose(1, 2)

        if past_kv is not None:
            past_k, past_v = past_kv
            k = torch.cat([past_k, k], dim=2)
            v = torch.cat([past_v, v], dim=2)

        L_kv = k.size(2)
        scores = torch.matmul(q, k.transpose(-2, -1)) / math.sqrt(self.head_dim)

        if bidirectional:
            if self.window_size is not None:
                # Respect local window boundaries even in bidirectional mode.
                # Without this, the window_size=2/4/8 branches see the full
                # sequence, destroying the multi-scale Inception receptive fields
                # for ~94% of training batches (all MLM batches).
                past_len = L_kv - L
                pos_i = (past_len + torch.arange(L, device=x.device)).unsqueeze(1)
                pos_j = torch.arange(L_kv, device=x.device).unsqueeze(0)
                dist  = torch.abs(pos_i - pos_j)  # symmetric for bidirectional
                win_mask = dist < self.window_size
                scores = scores.masked_fill(
                    ~win_mask.unsqueeze(0).unsqueeze(0), float('-inf')
                )
            # else: global bidirectional attention — no mask needed
        else:
            past_len = L_kv - L
            pos_i = (past_len + torch.arange(L, device=x.device)).unsqueeze(1)
            pos_j = torch.arange(L_kv, device=x.device).unsqueeze(0)
            dist  = pos_i - pos_j  # signed for causal (must be >= 0)

            causal_mask = dist >= 0
            if self.window_size is not None:
                causal_mask = causal_mask & (dist < self.window_size)

            scores = scores.masked_fill(
                ~causal_mask.unsqueeze(0).unsqueeze(0), float('-inf')
            )

        attn = torch.softmax(scores, dim=-1)
        out = torch.matmul(attn, v)                              # (B, H, L, head_dim)
        out = out.transpose(1, 2).contiguous().view(B, L, D)
        out = self.o_proj(out)

        if use_cache:
            if self.window_size is not None:
                present_kv = (
                    k[:, :, -self.window_size:, :],
                    v[:, :, -self.window_size:, :]
                )
            else:
                present_kv = (k, v)
        else:
            present_kv = None

        return out, present_kv


# ─────────────────────────────────────────────────────────────
# 2.  MSIT BRANCH BLOCK  (from MSIT codebase)
# ─────────────────────────────────────────────────────────────

class MSITBranchBlock(nn.Module):
    def __init__(self, dim: int, num_heads: int, window_size):
        super().__init__()
        self.ln1  = nn.LayerNorm(dim)
        self.attn = SlidingWindowAttention(dim, num_heads, window_size)
        self.ln2  = nn.LayerNorm(dim)
        self.ffn  = nn.Sequential(
            nn.Linear(dim, dim * 4, bias=False),
            nn.GELU(),
            nn.Linear(dim * 4, dim, bias=False),
        )

    def forward(self, x: torch.Tensor,
                past_kv=None,
                use_cache: bool = False,
                bidirectional: bool = False):
        attn_out, present_kv = self.attn(
            self.ln1(x), past_kv, use_cache, bidirectional
        )
        x = x + attn_out
        x = x + self.ffn(self.ln2(x))
        return x, present_kv


# ─────────────────────────────────────────────────────────────
# 3.  INCEPTION TRANSFORMER BLOCK  (from MSIT codebase)
# ─────────────────────────────────────────────────────────────

class InceptionTransformerBlock(nn.Module):
    def __init__(self, d_model: int = 512):
        super().__init__()
        assert d_model % 8 == 0, "d_model must be divisible by 8"

        self.dims    = [d_model // 2, d_model // 4, d_model // 8, d_model // 8]
        self.windows = [None, 8, 4, 2]
        self.heads   = [max(1, d // 64) for d in self.dims]

        self.proj_g = nn.Linear(d_model, self.dims[0], bias=False)
        self.proj_8 = nn.Linear(d_model, self.dims[1], bias=False)
        self.proj_4 = nn.Linear(d_model, self.dims[2], bias=False)
        self.proj_2 = nn.Linear(d_model, self.dims[3], bias=False)

        self.branch_g = MSITBranchBlock(self.dims[0], self.heads[0], self.windows[0])
        self.branch_8 = MSITBranchBlock(self.dims[1], self.heads[1], self.windows[1])
        self.branch_4 = MSITBranchBlock(self.dims[2], self.heads[2], self.windows[2])
        self.branch_2 = MSITBranchBlock(self.dims[3], self.heads[3], self.windows[3])

        self.out_proj   = nn.Linear(d_model, d_model, bias=False)
        self.layer_norm = nn.LayerNorm(d_model)

    def forward(self, x: torch.Tensor,
                past_kvs=None,
                use_cache: bool = False,
                bidirectional: bool = False):
        x_g = self.proj_g(x)
        x_8 = self.proj_8(x)
        x_4 = self.proj_4(x)
        x_2 = self.proj_2(x)

        pkv_g = past_kvs[0] if past_kvs else None
        pkv_8 = past_kvs[1] if past_kvs else None
        pkv_4 = past_kvs[2] if past_kvs else None
        pkv_2 = past_kvs[3] if past_kvs else None

        out_g, nkv_g = self.branch_g(x_g, pkv_g, use_cache, bidirectional)
        out_8, nkv_8 = self.branch_8(x_8, pkv_8, use_cache, bidirectional)
        out_4, nkv_4 = self.branch_4(x_4, pkv_4, use_cache, bidirectional)
        out_2, nkv_2 = self.branch_2(x_2, pkv_2, use_cache, bidirectional)

        concat = torch.cat([out_g, out_8, out_4, out_2], dim=-1)   # (B, T, d_model)
        out = self.layer_norm(x + self.out_proj(concat))

        present_kvs = (nkv_g, nkv_8, nkv_4, nkv_2) if use_cache else None
        return out, present_kvs


# ─────────────────────────────────────────────────────────────
# 4.  MODEL CONFIG
# ─────────────────────────────────────────────────────────────

@dataclass
class MSITGPTBERTConfig:
    vocab_size  : int   = VOCAB_SIZE   # 50257
    block_size  : int   = BLOCK_SIZE   # 512
    d_model     : int   = 512          # must be divisible by 8
    num_layers  : int   = 6
    dropout     : float = 0.1


# ─────────────────────────────────────────────────────────────
# 5.  FULL MSIT-GPT-BERT MODEL
# ─────────────────────────────────────────────────────────────

class MSITGPTBERTModel(nn.Module):
    def __init__(self, cfg: MSITGPTBERTConfig):
        super().__init__()
        self.cfg = cfg

        self.wte      = nn.Embedding(cfg.vocab_size, cfg.d_model)
        self.wpe      = nn.Embedding(cfg.block_size, cfg.d_model)
        self.drop_emb = nn.Dropout(cfg.dropout)

        self.blocks = nn.ModuleList([
            InceptionTransformerBlock(cfg.d_model)
            for _ in range(cfg.num_layers)
        ])

        self.ln_f    = nn.LayerNorm(cfg.d_model)
        self.lm_head = nn.Linear(cfg.d_model, cfg.vocab_size, bias=False)

        self.wte.weight = self.lm_head.weight

        self.apply(self._init_weights)
        total = sum(p.numel() for p in self.parameters())
        print(f"[MSIT-GPT-BERT] Total parameters: {total / 1e6:.2f}M")

    def _init_weights(self, module):
        if isinstance(module, nn.Linear):
            nn.init.normal_(module.weight, mean=0.0, std=0.02)
            if module.bias is not None:
                nn.init.zeros_(module.bias)
        elif isinstance(module, nn.Embedding):
            nn.init.normal_(module.weight, mean=0.0, std=0.02)

    def forward(self,
                input_ids: torch.Tensor,
                targets: torch.Tensor = None,
                bidirectional: bool = False):
        B, T = input_ids.size()
        assert T <= self.cfg.block_size, f"Sequence length {T} exceeds block_size {self.cfg.block_size}"

        pos = torch.arange(T, dtype=torch.long, device=input_ids.device)
        x   = self.drop_emb(self.wte(input_ids) + self.wpe(pos))

        for block in self.blocks:
            x, _ = block(x, past_kvs=None, use_cache=False, bidirectional=bidirectional)

        x      = self.ln_f(x)
        logits = self.lm_head(x)

        loss = None
        if targets is not None:
            loss = F.cross_entropy(
                logits.view(-1, self.cfg.vocab_size),
                targets.view(-1),
                ignore_index=-100
            )

        return logits, loss

    def configure_optimizers(self, weight_decay: float, learning_rate: float, device_type: str):
        param_dict     = {n: p for n, p in self.named_parameters() if p.requires_grad}
        decay_params   = [p for p in param_dict.values() if p.dim() >= 2]
        nodecay_params = [p for p in param_dict.values() if p.dim() < 2]
        groups = [
            {'params': decay_params,   'weight_decay': weight_decay},
            {'params': nodecay_params, 'weight_decay': 0.0},
        ]
        fused_ok  = 'fused' in inspect.signature(torch.optim.AdamW).parameters
        use_fused = fused_ok and ('cuda' in device_type)
        print(f"[Optimizer] fused AdamW: {use_fused}")
        return torch.optim.AdamW(groups, lr=learning_rate, betas=(0.9, 0.95), eps=1e-8, fused=use_fused)


# ─────────────────────────────────────────────────────────────
# 6.  DATA LOADER (UPGRADED WITH TIKTOKEN)
# ─────────────────────────────────────────────────────────────

class DataLoaderLite:
    """
    Overhauled DataLoader using tiktoken to eliminate single-document
    length context warnings while ensuring contiguous shard batching.
    """
    def __init__(self, B: int, T: int, texts: list, name: str = ""):
        self.B, self.T = B, T
        # Use tiktoken gpt2 matching tokenizer mapping properties
        enc = tiktoken.get_encoding("gpt2")
        self.eos_id = 50256 

        print(f"[DataLoader:{name}] Tokenising {len(texts)} documents using tiktoken ...")
        all_ids = []
        for t in texts:
            if t.strip():
                all_ids.extend(enc.encode(t, allowed_special={'<|endoftext|>'}))
                all_ids.append(self.eos_id)

        # Explicitly converting to long tensors for PyTorch operations
        self.tokens = torch.tensor(all_ids, dtype=torch.long)
        
        # Calculate chunks based on your batch dimension grid size
        self.chunk_size = B * T
        self.n_chunks = (len(self.tokens) - 1) // self.chunk_size
        self.indices = list(range(self.n_chunks))
        self.pos = 0
        self._shuffle()
        
        print(f"[DataLoader:{name}] Total tokens: {len(self.tokens):,} | "
              f"Iterations per Epoch: {self.n_chunks:,}")

    def _shuffle(self):
        random.shuffle(self.indices)
        self.pos = 0

    def steps_per_epoch(self) -> int:
        return self.n_chunks

    def next_batch(self):
        B, T = self.B, self.T
        if self.pos >= len(self.indices):
            self._shuffle()
            
        chunk_idx = self.indices[self.pos]
        self.pos += 1
        
        start_pos = chunk_idx * self.chunk_size
        # Take exactly (B * T + 1) tokens to construct shifted labels via a single view allocation
        temp = self.tokens[start_pos : start_pos + self.chunk_size + 1]
        
        x = temp[:-1].view(B, T)
        y = temp[1:].view(B, T)
        return x, y


# ─────────────────────────────────────────────────────────────
# 7.  HYBRID BATCH PREPARATION
# ─────────────────────────────────────────────────────────────

def get_current_mask_prob(global_step: int, total_steps: int) -> float:
    ratio = min(1.0, global_step / total_steps)
    return MASK_PROB_START + ratio * (MASK_PROB_END - MASK_PROB_START)


def prepare_causal_batch(x: torch.Tensor, y: torch.Tensor):
    return x, y, False


#  CORRECT MNTP ALIGNMENT
def prepare_masked_batch(x: torch.Tensor, y: torch.Tensor, mask_prob: float, mask_token_id: int):
    B, T = x.size()
    
    # 1. Generate the random boolean mask space relative to sequence indices
    mask = torch.rand(B, T, device=x.device) < mask_prob

    # 2. Mask the input sequence. If position k+1 is flagged, x[b, k] becomes [MASK]
    masked_x = x.clone()
    masked_x[mask] = mask_token_id

    # 3. Align the target matrix. The output at position k must predict token k+1.
    # y already contains the shifted values (y[b, k] == original token value at k+1)
    targets = torch.full_like(y, -100)
    targets[mask] = y[mask]

    return masked_x, targets, True

def get_hybrid_batch(train_loader: DataLoaderLite, global_step: int, total_steps: int, device: torch.device):
    x, y = train_loader.next_batch()
    x, y = x.to(device), y.to(device)

    if random.random() < CAUSAL_RATIO:
        input_ids, targets, bidir = prepare_causal_batch(x, y)
    else:
        mask_prob = get_current_mask_prob(global_step, total_steps)
        input_ids, targets, bidir = prepare_masked_batch(x, y, mask_prob, MASK_TOKEN_ID)

    return input_ids, targets, bidir


# ─────────────────────────────────────────────────────────────
# 8.  LEARNING RATE SCHEDULER
# ─────────────────────────────────────────────────────────────

def get_lr(it: int, total_steps: int) -> float:
    if it < WARMUP_STEPS:
        return LEARNING_RATE * (it + 1) / WARMUP_STEPS
    if it >= total_steps:
        return LR_MIN
    decay_ratio = (it - WARMUP_STEPS) / (total_steps - WARMUP_STEPS)
    coeff        = 0.5 * (1.0 + math.cos(math.pi * decay_ratio))
    return LR_MIN + coeff * (LEARNING_RATE - LR_MIN)


# ─────────────────────────────────────────────────────────────
# 9.  EVALUATION HELPERS
# ─────────────────────────────────────────────────────────────

#  DYNAMIC CONFIGURATION
@torch.inference_mode()
def sentence_log_prob(sentence: str, model: MSITGPTBERTModel, enc, device: torch.device, autocast_ctx, bidirectional: bool = False) -> float:
    ids = enc.encode(sentence, allowed_special={'<|endoftext|>'})
    if len(ids) <= 1:
        return -1e9
    ids = ids[:model.cfg.block_size]
    inp = torch.tensor([ids], dtype=torch.long, device=device)
    with autocast_ctx:
        # Respect the bidirectional evaluation flag passed from the runner loop
        logits, _ = model(inp, targets=None, bidirectional=bidirectional)
    shift_logits = logits[0, :-1, :]
    shift_labels = inp[0, 1:]
    lp = F.log_softmax(shift_logits, dim=-1)
    return lp[torch.arange(len(shift_labels)), shift_labels].sum().item()

def eval_blimp_folder(folder: str, model: MSITGPTBERTModel, enc, device: torch.device, autocast_ctx, tag: str = "BLiMP") -> float:
    if not os.path.exists(folder):
        print(f"  [WARN] '{folder}' not found — skipping {tag}.")
        return 0.0

    total, correct = 0, 0
    for fn in sorted(os.listdir(folder)):
        if not fn.endswith('.jsonl'):
            continue
        with open(os.path.join(folder, fn), encoding='utf-8') as f:
            for line in f:
                item = json.loads(line)
                sg = item.get('sentence_good')
                sb = item.get('sentence_bad')
                if sg and sb:
                    lp_g = sentence_log_prob(sg, model, enc, device, autocast_ctx,bidirectional=True)
                    lp_b = sentence_log_prob(sb, model, enc, device, autocast_ctx,bidirectional=True)
                    correct += int(lp_g > lp_b)
                    total   += 1

    acc = 100.0 * correct / total if total > 0 else 0.0
    print(f"  [{tag}] {correct}/{total} = {acc:.2f}%")
    return acc


def run_mid_epoch_eval(model: MSITGPTBERTModel, enc, device: torch.device, autocast_ctx, epoch: int, half: int):
    model.eval()
    label = f"Epoch {epoch+1}  " + ("— FIRST HALF ↓" if half == 0 else "— SECOND HALF ↓")
    print(f"\n{'='*65}\n  MID-EPOCH ZERO-SHOT EVAL  {label}\n{'='*65}")

    b_acc =0.00
    s_acc = eval_blimp_folder(SUPP_FAST_PATH, model, enc, device, autocast_ctx, tag="Supplement-fast")
    print(f"  ➜  BLiMP-fast: {b_acc:.2f}%   Supplement-fast: {s_acc:.2f}%\n{'='*65}\n")
    model.train()
    return b_acc, s_acc


@torch.inference_mode()
def evaluate_val_loss(model: MSITGPTBERTModel, val_loader: DataLoaderLite, device: torch.device, autocast_ctx, global_step: int, total_steps: int, eval_iters: int = 20) -> float:
    model.eval()

    # Snapshot the full loader state before eval, including any in-flight shuffle,
    # so a shuffle that fires mid-loop doesn't corrupt the restore.
    old_pos     = val_loader.pos
    old_indices = list(val_loader.indices)  # deep copy — list() is sufficient for int lists

    total = 0.0
    for _ in range(eval_iters):
        input_ids, targets, bidir = get_hybrid_batch(val_loader, global_step, total_steps, device)
        with autocast_ctx:
            _, loss = model(input_ids, targets, bidirectional=bidir)
        total += loss.item()

    # Restore unconditionally after the loop — covers the mid-loop shuffle case.
    val_loader.indices = old_indices
    val_loader.pos     = old_pos

    model.train()
    return total / eval_iters


# ─────────────────────────────────────────────────────────────
# 10.  MAIN TRAINING LOOP
# ─────────────────────────────────────────────────────────────

def main():
    torch.manual_seed(42)
    random.seed(42)

    device       = 'cuda' if torch.cuda.is_available() else 'cpu'
    amp_dtype    = torch.bfloat16
    autocast_ctx = torch.autocast(device_type=device, dtype=amp_dtype, enabled=(device == 'cuda'))
    
    if device == 'cuda':
        torch.set_float32_matmul_precision('high')
        torch.cuda.manual_seed(42)

    print(f"[Tokenizer] Loading tiktoken gpt2 encoder backend...")
    enc = tiktoken.get_encoding("gpt2")

    print("\n[Data] Loading BabyLM-2026-Strict-Small ...")
    ds       = load_dataset("BabyLM-community/BabyLM-2026-Strict-Small")
    all_text = list(ds['train']['text'])

    split       = int(len(all_text) * 0.95)
    train_texts = all_text[:split]
    val_texts   = all_text[split:]

    train_loader = DataLoaderLite(BATCH_SIZE, BLOCK_SIZE, train_texts, "train")
    val_loader   = DataLoaderLite(BATCH_SIZE, BLOCK_SIZE, val_texts, "val")

    steps_per_epoch = train_loader.steps_per_epoch()
    total_steps     = steps_per_epoch * EPOCHS
    half_step       = steps_per_epoch // 2

    cfg   = MSITGPTBERTConfig()
    model = MSITGPTBERTModel(cfg).to(device)

    try:
        model = torch.compile(model)
        print("[Model] torch.compile() successfully verified")
    except Exception as e:
        print(f"[Model] torch.compile() skipped ({e})")

    optimizer = model.configure_optimizers(WEIGHT_DECAY, LEARNING_RATE, device)
    os.makedirs("checkpoints", exist_ok=True)

    model.train()
    global_step = 0
    best_val    = float('inf')

    for epoch in range(EPOCHS):
        train_loader._shuffle()
        first_half_done  = False
        second_half_done = False

        for step in range(steps_per_epoch):
            t0 = time.perf_counter()

            lr = get_lr(global_step, total_steps)
            for pg in optimizer.param_groups:
                pg['lr'] = lr

            optimizer.zero_grad(set_to_none=True)
            loss_accum = 0.0

            for _ in range(GRAD_ACCUM_STEPS):
                input_ids, targets, bidir = get_hybrid_batch(train_loader, global_step, total_steps, device)
                with autocast_ctx:
                    _, loss = model(input_ids, targets, bidirectional=bidir)
                scaled_loss  = loss / GRAD_ACCUM_STEPS
                loss_accum  += scaled_loss.item()
                scaled_loss.backward()

            norm = torch.nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP)
            optimizer.step()

            if device == 'cuda':
                torch.cuda.synchronize()

            dt  = (time.perf_counter() - t0) * 1000
            tps = (BATCH_SIZE * BLOCK_SIZE * GRAD_ACCUM_STEPS) / (dt / 1000)

            val_str = ""
            if global_step % 50 == 0:
                vl = evaluate_val_loss(model, val_loader, device, autocast_ctx, global_step, total_steps, eval_iters=20)
                val_str = f"  val={vl:.4f}"
                if vl < best_val:
                    best_val = vl
                    raw = model._orig_mod if hasattr(model, '_orig_mod') else model
                    torch.save({
                        'step':                 global_step,
                        'epoch':                epoch,
                        'model_state_dict':     raw.state_dict(),
                        'optimizer_state_dict': optimizer.state_dict(),  # required for clean resume
                        'config':               cfg,
                        'val_loss':             vl,
                    }, "checkpoints/msit_gptbert_best.pt")

            mode_tag = "MLM" if bidir else "CLM"
            mask_p   = get_current_mask_prob(global_step, total_steps)
            print(
                f"[E{epoch+1:02d} {step+1:>5d}/{steps_per_epoch} G{global_step:>7d}|{mode_tag}] "
                f"train={loss_accum:.4f}{val_str}  mask={mask_p:.1%}  norm={norm:.3f}  lr={lr:.2e}  dt={dt:6.1f}ms"
            )

            if (step + 1) == half_step and not first_half_done:
                first_half_done = True
                run_mid_epoch_eval(model, enc, device, autocast_ctx, epoch, half=0)

            if (step + 1) == steps_per_epoch and not second_half_done:
                second_half_done = True
                run_mid_epoch_eval(model, enc, device, autocast_ctx, epoch, half=1)

            global_step += 1

        raw  = model._orig_mod if hasattr(model, '_orig_mod') else model
        torch.save({
            'epoch':                epoch + 1,
            'step':                 global_step,
            'model_state_dict':     raw.state_dict(),
            'optimizer_state_dict': optimizer.state_dict(),
            'config':               cfg,
        }, f"checkpoints/msit_gptbert_epoch{epoch+1:02d}.pt")


if __name__ == "__main__":
    main()

[Tokenizer] Loading tiktoken gpt2 encoder backend...

[Data] Loading BabyLM-2026-Strict-Small ...


README.md:   0%|          | 0.00/2.56k [00:00<?, ?B/s]

bnc_spoken.train.txt:   0%|          | 0.00/4.06M [00:00<?, ?B/s]

childes.train.txt:   0%|          | 0.00/15.2M [00:00<?, ?B/s]

gutenberg.train.txt:   0%|          | 0.00/14.0M [00:00<?, ?B/s]

open_subtitles.train.txt:   0%|          | 0.00/12.2M [00:00<?, ?B/s]

simple_wiki.train.txt:   0%|          | 0.00/8.86M [00:00<?, ?B/s]

switchboard.train.txt:   0%|          | 0.00/123k [00:00<?, ?B/s]

Generating train split:   0%|          | 0/1104106 [00:00<?, ? examples/s]

[DataLoader:train] Tokenising 1048900 documents using tiktoken ...
[DataLoader:train] Total tokens: 14,561,988 | Iterations per Epoch: 1,777
[DataLoader:val] Tokenising 55206 documents using tiktoken ...
[DataLoader:val] Total tokens: 1,941,013 | Iterations per Epoch: 236
[MSIT-GPT-BERT] Total parameters: 35.65M
[Model] torch.compile() successfully verified
[Optimizer] fused AdamW: True
[E01     1/1777 G      0|MLM] train=10.6291  val=10.8646  mask=30.0%  norm=9.462  lr=3.75e-07  dt=24686.3ms
[E01     2/1777 G      1|MLM] train=10.5748  mask=30.0%  norm=7.047  lr=7.50e-07  dt=  29.1ms
[E01     3/1777 G      2|MLM] train=10.5784  mask=30.0%  norm=6.878  lr=1.13e-06  dt=  24.4ms
[E01     4/1777 G      3|MLM] train=10.5415  mask=30.0%  norm=6.897  lr=1.50e-06  dt=  24.0ms
[E01     5/1777 G      4|MLM] train=10.8552  mask=30.0%  norm=4.776  lr=1.87e-06  dt=  24.0ms
[E01     6/1777 G      5|MLM] train=10.5361  mask=30.0%  norm=10.389  lr=2.25e-06  dt=  24.0ms
[E01     7/1777 G      6|MLM] t